# 06 – Nevralt nettverk, evaluering og eksperimenter

**Tid:** 180 minutter: 60 minutter modell og repetisjon, 75 minutter
evaluering og eksperiment, 45 minutter selvstendig oppsummering.

**Mål:** legge til et skjult lag, undersøke feil og sammenligne to modeller
med ellers like innstillinger. Vi bruker fortsatt ingen konvolusjonslag.


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import torch
from torch import nn


In [ ]:
from pytorch_101.utils import download_mnist

treningsdata, testdata = download_mnist()
print(f"Treningsbilder: {len(treningsdata)}, testbilder: {len(testdata)}")


## Eksempel: et skjult lag og ReLU

Et skjult lag lager mellomliggende representasjoner. ReLU erstatter negative
verdier med 0 og beholder positive. Uten en ikke-lineær aktivering ville flere
lineære lag samlet fortsatt bare vært én lineær transformasjon.

Her er et lite eksempel med tre innganger, fire skjulte nevroner og to utganger.
Det siste laget gir rå skårer; vi legger ikke ReLU eller softmax etter det.


In [ ]:
liten_modell = nn.Sequential(nn.Linear(3, 4), nn.ReLU(), nn.Linear(4, 2))
print(liten_modell(torch.ones(5, 3)).shape)
verdier = torch.tensor([-2., -1., 0., 1., 2.])
print("Før ReLU:", verdier, "Etter:", nn.ReLU()(verdier))


## Oppgave 1: Lag MNIST-nettverket

Skriv funksjonen `lag_modell(antall_skjulte)` som returnerer:
`Flatten → Linear(784, antall_skjulte) → ReLU → Linear(antall_skjulte, 10)`.
Opprett `modell = lag_modell(128)`. Vis formen etter hvert lag for en gruppe
på 32 nullbilder, og tell modellens parametere.

<details>
<summary>Hint</summary>

Bruk en vanlig funksjon som returnerer `nn.Sequential`. Iterer over lagene for å vise mellomformene.

</details>


In [ ]:
def lag_modell(antall_skjulte):
    return nn.Sequential(
        nn.Flatten(),
        nn.Linear(784, antall_skjulte),
        nn.ReLU(),
        nn.Linear(antall_skjulte, 10),
    )

modell = lag_modell(128)
verdier = torch.zeros(32, 1, 28, 28)
print("Inndata:", verdier.shape)
for lag in modell:
    verdier = lag(verdier)
    print(type(lag).__name__, verdier.shape)
print("Parametere:", sum(parameter.numel() for parameter in modell.parameters()))


**Kontroll:** Kjør denne cellen når du har løst oppgaven. En feilmelding betyr at du må undersøke løsningen.


In [ ]:
assert modell(torch.zeros(32, 1, 28, 28)).shape == (32, 10)


### Sjekk forståelsen

Skriv formene gjennom modellen med 128 skjulte nevroner.
Hva tilfører ReLU? Hvorfor har siste lag ti utganger, uansett størrelsen på det skjulte laget?

Skriv svaret ditt i Markdown-cellen under. Bruk egne resultater der det er relevant.


**Mitt svar:**

`[32, 1, 28, 28] → [32, 784] → [32, 128] → [32, 128] → [32, 10]`.
ReLU tilfører ikke-linearitet uten å endre formen. Den lar modellen lære
andre sammenhenger enn en samlet lineær transformasjon. De ti utgangene
tilsvarer de ti mulige sifrene, ikke antall skjulte nevroner.


## Oppgave 2: Gjenbruk og forstå treningsløkken

Gjør treningskoden din fra notatbok 05 om til funksjonen
`tren_modell(modell, treningslaster, antall_epoker=5, laeringsrate=0.1)`.

Funksjonen skal opprette kryssentropitap og en ny SGD-optimizer, trene modellen
og returnere en liste med gjennomsnittlig treningstap per epoke. Antall
treningsbilder finnes med `len(treningslaster.dataset)`.

Behold de fem treningsstegene synlige. Skriv ut tapet etter hver epoke.
Vi definerer funksjonen nå og kjører den i sammenligningen under.

<details>
<summary>Hint</summary>

Flytt løkken inn i en funksjon. Opprett optimizer inne i funksjonen slik at den tilhører modellen som sendes inn.

</details>


In [ ]:
def tren_modell(modell, treningslaster, antall_epoker=5, laeringsrate=0.1):
    tapsfunksjon = nn.CrossEntropyLoss()
    optimizer = torch.optim.SGD(modell.parameters(), lr=laeringsrate)
    historikk = []
    for epoke in range(antall_epoker):
        modell.train()
        tapsum = 0.0
        for bilder, etiketter in treningslaster:
            optimizer.zero_grad()
            skaarer = modell(bilder)
            tap = tapsfunksjon(skaarer, etiketter)
            tap.backward()
            optimizer.step()
            tapsum += tap.item() * len(bilder)
        historikk.append(tapsum / len(treningslaster.dataset))
        print(f"Epoke {epoke + 1}: treningstap {historikk[-1]:.4f}")
    return historikk


## Oppgave 3: Samle evalueringsresultater

Skriv `evaluer_modell(modell, datalaster)` ved å tilpasse testløkken fra
notatbok 05. Returner en ordbok med:

- `noyaktighet`: antall riktige delt på antall bilder.
- `prediksjoner`: alle predikerte sifre, samlet i én tensor.
- `etiketter`: alle riktige sifre, samlet i samme rekkefølge.

Bruk `eval()` og `no_grad()`. Funksjonen skal ikke oppdatere modellen.

<details>
<summary>Hint</summary>

Samle tensorer i to lister og bruk `torch.cat` til slutt. Bruk `argmax(dim=1)` for prediksjoner.

</details>


In [ ]:
def evaluer_modell(modell, datalaster):
    modell.eval()
    alle_prediksjoner = []
    alle_etiketter = []
    with torch.no_grad():
        for bilder, etiketter in datalaster:
            skaarer = modell(bilder)
            alle_prediksjoner.append(skaarer.argmax(dim=1))
            alle_etiketter.append(etiketter)
    prediksjoner = torch.cat(alle_prediksjoner)
    etiketter = torch.cat(alle_etiketter)
    noyaktighet = (prediksjoner == etiketter).float().mean().item()
    return {"noyaktighet": noyaktighet, "prediksjoner": prediksjoner, "etiketter": etiketter}


## Et kontrollert eksperiment

Vi sammenligner **32 og 128 skjulte nevroner**, valgt før vi ser testresultatene.
Hold datasett, gruppestørrelse, fem epoker og læringsrate 0.1 like.
I vanlige oppgaver har vi latt tilfeldighetene variere. Nå bruker vi
`seed_everything(42)` fra `pytorch_101.utils` før hvert forsøk. Den setter frø for
Python, NumPy og PyTorch, slik at forsøket lettere kan gjentas.

Opprett nye datalastere direkte i løkken. Gi treningslasteren en egen
`torch.Generator().manual_seed(42)` for samme stokking i begge forsøkene.
Dette er frøet til én laster, ikke global frøsetting. Modellene har ulike
former og får derfor ikke identiske startvekter selv med samme frø.

Testsettet brukes til denne avtalte rapporteringen. Ikke fortsett å velge
innstillinger etter testresultatene. Systematisk tuning krever et separat
valideringssett, som vi utelater i dette introduksjonskurset.


In [ ]:
from time import perf_counter
from torch.utils.data import DataLoader
from pytorch_101.utils import seed_everything

# Eksempel på tidtaking; bruk samme mønster rundt selve treningen.
start = perf_counter()
sum(range(10_000))
print(f"Eksempelberegningen tok {perf_counter() - start:.6f} sekunder")


## Oppgave 4: Sammenlign modellene

Lag ordbøkene `modeller`, `historikker`, `resultater` og `treningstider`,
med antall skjulte nevroner som nøkkel. For hver størrelse:

1. Kall `seed_everything(42)`, opprett ny modell og nye `DataLoader`-objekter
   med `batch_size=64`. Stokk treningsdata med en egen generator med frø 42;
   behold testrekkefølgen.
2. Mål tiden for trening og lagre tapshistorikken.
3. Evaluer én gang på testutvalget og lagre resultatet og modellen.

Skriv ut en tabell med skjult størrelse, siste treningstap, testnøyaktighet og
treningstid. Tegn begge treningskurvene i én figur med norske tekster.

<details>
<summary>Hint</summary>

Bruk funksjonene over og en løkke over `[32, 128]`. Ikke tren den samme modellen videre mellom forsøkene.

</details>


In [ ]:
modeller = {}
historikker = {}
resultater = {}
treningstider = {}
for antall_skjulte in [32, 128]:
    seed_everything(42)
    modell = lag_modell(antall_skjulte)
    treningslaster = DataLoader(
        treningsdata, batch_size=64, shuffle=True,
        generator=torch.Generator().manual_seed(42),
    )
    testlaster = DataLoader(testdata, batch_size=64, shuffle=False)
    start = perf_counter()
    historikker[antall_skjulte] = tren_modell(modell, treningslaster)
    treningstider[antall_skjulte] = perf_counter() - start
    resultater[antall_skjulte] = evaluer_modell(modell, testlaster)
    modeller[antall_skjulte] = modell

print("Skjulte | Siste treningstap | Testnøyaktighet | Treningstid")
for antall_skjulte in [32, 128]:
    print(f"{antall_skjulte:7} | {historikker[antall_skjulte][-1]:16.4f} | "
          f"{resultater[antall_skjulte]['noyaktighet']:15.1%} | {treningstider[antall_skjulte]:.2f} s")
figur, akse = plt.subplots()
for antall_skjulte in [32, 128]:
    akse.plot(range(1, 6), historikker[antall_skjulte], marker="o", label=f"{antall_skjulte} nevroner")
akse.set(xlabel="Epoke", ylabel="Gjennomsnittlig treningstap", title="To størrelser på det skjulte laget")
akse.legend()
plt.show()


**Kontroll:** Kjør denne cellen når du har løst oppgaven. En feilmelding betyr at du må undersøke løsningen.


In [ ]:
assert set(modeller) == {32, 128}
for storrelse in [32, 128]:
    assert len(historikker[storrelse]) == 5
    assert np.isfinite(historikker[storrelse]).all()
    assert historikker[storrelse][-1] < historikker[storrelse][0]
    assert resultater[storrelse]["prediksjoner"].shape == (1000,)
    assert 0 <= resultater[storrelse]["noyaktighet"] <= 1


### Sjekk forståelsen

Skriv inn målt sluttap, testnøyaktighet og treningstid for begge modeller.
Hvilke forskjeller ser du? Kan ett forsøk bevise at flere nevroner alltid er bedre?
Kan vi avgjøre om modellen overtilpasser bare ved å se at treningstapet synker?

Skriv svaret ditt i Markdown-cellen under. Bruk egne resultater der det er relevant.


**Mitt svar:**

I eksempelkjøringen fikk 32-modellen sluttap 0.3376, testnøyaktighet 92,6 %
og treningstid 0,61 sekunder. 128-modellen fikk sluttap 0.3266,
testnøyaktighet 92,6 % og treningstid 0,65 sekunder. Tidene avhenger av maskinen;
bruk din egen måling. Den største modellen fikk litt lavere treningstap,
men samme testnøyaktighet. Flere nevroner ga altså ikke bedre testresultat her.
Ett frø og et lite
testutvalg gir begrenset grunnlag for en generell konklusjon.
Synkende treningstap viser tilpasning til treningsdata, men alene sier det
ikke om overtilpasning. Det krever sammenligning med data utenfor treningen,
normalt et valideringssett under modellutviklingen.


## Eksempel: finn posisjoner som oppfyller en betingelse

`torch.where` kan finne posisjonene der to tensorer er forskjellige.
Dette bruker vi til å finne feilklassifiserte bilder.


In [ ]:
eksempelgjett = torch.tensor([1, 2, 3, 4])
eksempelfasit = torch.tensor([1, 7, 3, 9])
feilposisjoner = torch.where(eksempelgjett != eksempelfasit)[0]
print("Feil på posisjonene:", feilposisjoner.tolist())


## Oppgave 5: Vis modellens feil

Bruk den forhåndsvalgte modellen med **128 skjulte nevroner**.
Finn feilindeksene fra `resultater[128]` og vis opptil ni feilklassifiserte
testbilder. Skriv riktig og predikert siffer i tittelen for hvert bilde.
Håndter også at modellen kan ha færre enn ni feil: skjul ubrukte akser.

`testlaster` stokker ikke, så resultatene har samme rekkefølge som `testdata`.

<details>
<summary>Hint</summary>

Kombiner `torch.where` med rutenettet fra notatbok 02. Bruk `indeks.item()` når du indekserer datasettet.

</details>


In [ ]:
prediksjoner = resultater[128]["prediksjoner"]
etiketter = resultater[128]["etiketter"]
feilindekser = torch.where(prediksjoner != etiketter)[0][:9]
figur, akser = plt.subplots(3, 3, figsize=(8, 8))
for akse in akser.ravel():
    akse.axis("off")
for akse, indeks in zip(akser.ravel(), feilindekser):
    posisjon = indeks.item()
    bilde, etikett = testdata[posisjon]
    akse.imshow(bilde.squeeze(0), cmap="gray", vmin=0, vmax=1)
    akse.set_title(f"Riktig: {etikett}, gjettet: {prediksjoner[posisjon].item()}")
figur.suptitle("Eksempler på feilklassifiseringer")
figur.tight_layout()
plt.show()
print("Antall feil i hele testutvalget:", (prediksjoner != etiketter).sum().item())


### Sjekk forståelsen

Beskriv to konkrete feil fra figuren. Ser noen sifre like ut?
Er ni utvalgte feil nok til å si hvilken klasse modellen er dårligst på?
Hva måtte vi undersøkt for å svare bedre på det?

Skriv svaret ditt i Markdown-cellen under. Bruk egne resultater der det er relevant.


**Mitt svar:**

I eksempelkjøringen ble det første feilbildet, en 5-er, tolket som 3.
De tydelige øvre og nedre strekene kan ligne delene i en 3-er.
Det neste feilbildet, en 6-er, ble tolket som 0; den runde nedre delen
kan bidra til forvekslingen. Dette er mulige visuelle forklaringer,
ikke et bevis på hvilke trekk modellen brukte. De ni første feilene er ikke en fullstendig
analyse. Vi måtte telt feil og antall eksempler per klasse for hele utvalget,
eventuelt laget en forvekslingsmatrise og vurdert usikkerhet ved få eksempler.


## Ferdig eksempel: vis sannsynligheter for ett bilde

Softmax omformer skårene til positive tall som summerer til 1.
Dette er modellens beregnede sannsynligheter, ikke en garanti for at den har rett.
Vi bruker softmax til visning, aldri før kryssentropitapet.


In [ ]:
bilde, etikett = testdata[0]
modeller[128].eval()
with torch.no_grad():
    skaarer = modeller[128](bilde.unsqueeze(0))
    sannsynligheter = skaarer.softmax(dim=1).squeeze(0)
figur, akse = plt.subplots()
akse.bar(range(10), sannsynligheter.numpy())
akse.set_xticks(range(10))
akse.set(xlabel="Siffer", ylabel="Beregnet sannsynlighet", title=f"Riktig siffer: {etikett}")
plt.show()


## Selvstendig oppsummering – 45 minutter

Her skal du sette sammen delene med mindre støtte. Du kan slå opp tidligere
eksempler, men skriv koden og forklaringene selv. Arbeid gjerne to og to,
med hver deres skriftlige svar.


## Oppgave 6: Bygg og forklar hele kjeden

Lag en **ny** lineær modell med `Flatten` og ett lineært lag.
Kall den `egen_modell`. Opprett nye datalastere direkte med `DataLoader` (64 bilder per gruppe,
trening med stokking og test uten stokking) og
en ny SGD-optimizer med læringsrate 0.1. Tren modellen i én epoke og evaluer
den på testutvalget. Skriv både treningsløkken og evalueringsløkken direkte
i cellen; ikke kall `tren_modell` eller `evaluer_modell`.

Skriv ut gjennomsnittlig treningstap og testnøyaktighet. Denne korte repetisjonen
er ikke en rettferdig ytelsessammenligning med modellene som fikk fem epoker.

<details>
<summary>Hint</summary>

Skriv først rekkefølgen med egne ord: data → skårer → tap → gradienter → oppdatering. Legg så til evaluering uten oppdatering.

</details>


In [ ]:
egen_modell = nn.Sequential(nn.Flatten(), nn.Linear(784, 10))
egen_treningslaster = DataLoader(treningsdata, batch_size=64, shuffle=True)
egen_testlaster = DataLoader(testdata, batch_size=64, shuffle=False)
egen_optimizer = torch.optim.SGD(egen_modell.parameters(), lr=0.1)
egen_tapsfunksjon = nn.CrossEntropyLoss()
egen_modell.train()
tapsum = 0.0
for bilder, etiketter in egen_treningslaster:
    egen_optimizer.zero_grad()
    skaarer = egen_modell(bilder)
    tap = egen_tapsfunksjon(skaarer, etiketter)
    tap.backward()
    egen_optimizer.step()
    tapsum += tap.item() * len(bilder)
print(f"Treningstap: {tapsum / len(treningsdata):.4f}")

egen_modell.eval()
antall_riktige = 0
with torch.no_grad():
    for bilder, etiketter in egen_testlaster:
        prediksjoner = egen_modell(bilder).argmax(dim=1)
        antall_riktige += (prediksjoner == etiketter).sum().item()
print(f"Testnøyaktighet: {antall_riktige / len(testdata):.1%}")


### Sjekk forståelsen

Forklar hele kjeden fra en gruppe bilder til oppdaterte parametere.
Ta med tensorformene, tapsfunksjonens rolle og forskjellen mellom gradientberegning
og parameteroppdatering. Forklar også hvorfor evaluering er en separat løkke.

Skriv svaret ditt i Markdown-cellen under. Bruk egne resultater der det er relevant.


**Mitt svar:**

En gruppe `[B, 1, 28, 28]` flates til `[B, 784]` og blir til rå skårer `[B, 10]`.
Kryssentropitapet sammenligner skårene med etiketter `[B]` og gir ett tapsmål.
Vi nullstiller gamle gradienter, og `backward()` beregner nye gradienter for
modellens parametere. `step()` oppdaterer parametrene i retning av lavere tap.
Evalueringen bruker egne testbilder uten gradientberegning eller oppdateringer,
slik at vi måler prediksjoner på bilder modellen ikke lærte fra.


## Valgfri ekstraoppgave

Lag en forvekslingsmatrise for den ferdigtrente 128-modellen: rader er riktige
sifre, kolonner er predikerte sifre. Vis den med `imshow`, fargeskala og norske
aksetekster. Undersøk hvilke forvekslinger som er vanligst. Bruk eksisterende
prediksjoner; ikke tren eller velg nye modeller ut fra testresultatet.


In [ ]:
matrise = torch.zeros(10, 10, dtype=torch.int64)
for riktig, predikert in zip(resultater[128]["etiketter"], resultater[128]["prediksjoner"]):
    matrise[riktig, predikert] += 1
figur, akse = plt.subplots(figsize=(7, 6))
visning = akse.imshow(matrise.numpy(), cmap="Blues")
figur.colorbar(visning, ax=akse, label="Antall bilder")
akse.set(xlabel="Predikert siffer", ylabel="Riktig siffer", title="Forvekslingsmatrise")
akse.set_xticks(range(10))
akse.set_yticks(range(10))
plt.show()
